In [1]:
!pip -q install PyPDF2 python-docx reportlab

print("ติดตั้งไลบรารีเรียบร้อยแล้ว")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.1 MB/s eta 0:00:00
ติดตั้งไลบรารีเรียบร้อยแล้ว


In [2]:
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
import os

BASE_DIR = Path("/content/document-automator")
SRC_DIR = BASE_DIR / "src"
DOCS_DIR = BASE_DIR / "documents"

SRC_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

(SRC_DIR / "__init__.py").write_text("", encoding="utf-8")


def create_sample_pdf(filepath, pages):
    """สร้าง PDF จากรายการข้อความ โดยแต่ละรายการเป็นหนึ่งหน้า"""
    pdf = canvas.Canvas(str(filepath), pagesize=A4)
    page_width, page_height = A4

    for lines in pages:
        y = page_height - 70

        for line in lines:
            pdf.drawString(60, y, line)
            y -= 24

        pdf.showPage()

    pdf.save()


article_pages = [
    [
        "Introduction to Automation",
        "Automation uses technology to perform tasks",
        "with minimal human intervention.",
        "It improves efficiency, reliability, and speed.",
        "Automation can increase productivity, reduce costs,",
        "and improve the quality of work."
    ],
    [
        "Benefits of Automation",
        "1. Increased Efficiency: Tasks are completed faster.",
        "2. Cost Reduction: Lower labor costs and less waste.",
        "3. Improved Accuracy: Minimizes human error.",
        "4. Enhanced Safety: Automates dangerous tasks.",
        "5. Greater Consistency: Provides uniform quality.",
        "",
        "Conclusion",
        "Automation is transforming industries globally."
    ]
]

create_sample_pdf(DOCS_DIR / "input_article.pdf", article_pages)

create_sample_pdf(
    DOCS_DIR / "dummy_doc1.pdf",
    [["Dummy PDF 1", "This is the first sample document."]]
)

create_sample_pdf(
    DOCS_DIR / "dummy_doc2.pdf",
    [["Dummy PDF 2", "This is the second sample document."]]
)

print("สร้างไฟล์ PDF ตัวอย่างเรียบร้อย")
for path in sorted(DOCS_DIR.glob("*.pdf")):
    print(path.name, "-", path.stat().st_size, "bytes")

สร้างไฟล์ PDF ตัวอย่างเรียบร้อย
dummy_doc1.pdf - 1407 bytes
dummy_doc2.pdf - 1408 bytes
input_article.pdf - 2331 bytes


In [3]:
%%writefile /content/document-automator/src/utils.py
import logging
import os


def setup_logging(name, level=logging.INFO):
    """สร้าง logger สำหรับแต่ละโมดูล"""
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False

    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger


def ensure_directory_exists(path):
    """สร้างโฟลเดอร์ หากยังไม่มี"""
    try:
        os.makedirs(path, exist_ok=True)
    except OSError as error:
        logging.error("Cannot create directory %s: %s", path, error)
        raise

Writing /content/document-automator/src/utils.py


In [4]:
%%writefile /content/document-automator/src/pdf_processor.py
import os
from pathlib import Path

from PyPDF2 import PdfReader, PdfWriter
from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


class PDFProcessor:
    """จัดการงาน PDF ด้วย PyPDF2"""

    def __init__(self):
        logger.info("PDFProcessor initialized.")

    def extract_text_from_pdf(self, pdf_path):
        """ดึงข้อความจากทุกหน้าของ PDF"""
        if not os.path.exists(pdf_path):
            logger.error("PDF file not found: %s", pdf_path)
            return None

        try:
            reader = PdfReader(pdf_path)

            if reader.is_encrypted:
                logger.error("PDF is encrypted: %s", pdf_path)
                return None

            text_parts = []

            for page_num, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""
                text_parts.append(
                    f"{page_text}\n--- Page End: {page_num} ---"
                )

            text_content = "\n\n".join(text_parts)
            logger.info(
                "Extracted text from %s (%d pages).",
                pdf_path, len(reader.pages)
            )
            return text_content

        except Exception as error:
            logger.exception("Error extracting PDF text: %s", error)
            return None

    def merge_pdfs(self, pdf_paths, output_path):
        """รวม PDF หลายไฟล์เป็นไฟล์เดียว"""
        if not pdf_paths:
            logger.warning("No PDF paths provided.")
            return False

        writer = PdfWriter()
        added_pages = 0

        try:
            for pdf_path in pdf_paths:
                if not os.path.exists(pdf_path):
                    logger.warning("Skipping missing file: %s", pdf_path)
                    continue

                reader = PdfReader(pdf_path)

                if reader.is_encrypted:
                    logger.warning("Skipping encrypted PDF: %s", pdf_path)
                    continue

                for page in reader.pages:
                    writer.add_page(page)
                    added_pages += 1

            if added_pages == 0:
                logger.warning("No pages available to merge.")
                return False

            output_path = Path(output_path)
            ensure_directory_exists(str(output_path.parent))

            with open(output_path, "wb") as output_file:
                writer.write(output_file)

            logger.info("Merged %d pages into %s", added_pages, output_path)
            return True

        except Exception as error:
            logger.exception("Error merging PDFs: %s", error)
            return False

        finally:
            writer.close()

    def split_pdf(self, pdf_path, output_dir=".", pages_per_split=1):
        """แบ่ง PDF เป็นหลายไฟล์ตามจำนวนหน้าต่อไฟล์"""
        if not os.path.exists(pdf_path):
            logger.error("PDF file not found: %s", pdf_path)
            return False

        if not isinstance(pages_per_split, int) or pages_per_split < 1:
            logger.error("pages_per_split must be a positive integer.")
            return False

        try:
            reader = PdfReader(pdf_path)

            if reader.is_encrypted:
                logger.error("Cannot split encrypted PDF: %s", pdf_path)
                return False

            ensure_directory_exists(output_dir)

            base_name = Path(pdf_path).stem
            total_pages = len(reader.pages)

            for start in range(0, total_pages, pages_per_split):
                writer = PdfWriter()
                end = min(start + pages_per_split, total_pages)

                for page_index in range(start, end):
                    writer.add_page(reader.pages[page_index])

                output_file = Path(output_dir) / (
                    f"{base_name}_part_{start // pages_per_split + 1}.pdf"
                )

                with open(output_file, "wb") as file:
                    writer.write(file)

                writer.close()
                logger.info("Created split PDF: %s", output_file)

            return True

        except Exception as error:
            logger.exception("Error splitting PDF: %s", error)
            return False

    def rotate_pdf(self, pdf_path, output_path, degrees=90):
        """หมุนทุกหน้าใน PDF แล้วบันทึกเป็นไฟล์ใหม่"""
        if degrees not in (90, 180, 270):
            logger.error("Rotation must be 90, 180, or 270 degrees.")
            return False

        try:
            reader = PdfReader(pdf_path)
            writer = PdfWriter()

            if reader.is_encrypted:
                logger.error("Cannot rotate encrypted PDF.")
                return False

            for page in reader.pages:
                page.rotate(degrees)
                writer.add_page(page)

            ensure_directory_exists(str(Path(output_path).parent))

            with open(output_path, "wb") as file:
                writer.write(file)

            writer.close()
            logger.info("Rotated PDF saved to %s", output_path)
            return True

        except Exception as error:
            logger.exception("Error rotating PDF: %s", error)
            return False

Writing /content/document-automator/src/pdf_processor.py


In [5]:
%%writefile /content/document-automator/src/word_processor.py
import os
import docx

from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


class WordProcessor:
    """จัดการเอกสาร Word ด้วย python-docx"""

    def __init__(self):
        logger.info("WordProcessor initialized.")

    def create_document(self):
        return docx.Document()

    def add_heading(self, document, text, level=1):
        document.add_heading(text, level=level)

    def add_paragraph(
        self, document, text, style=None, bold=False,
        italic=False, font_size=None, align="left"
    ):
        paragraph = document.add_paragraph(style=style)
        run = paragraph.add_run(str(text))

        run.bold = bold
        run.italic = italic

        if font_size is not None:
            run.font.size = Pt(font_size)

        alignment_map = {
            "left": WD_ALIGN_PARAGRAPH.LEFT,
            "center": WD_ALIGN_PARAGRAPH.CENTER,
            "right": WD_ALIGN_PARAGRAPH.RIGHT,
            "justify": WD_ALIGN_PARAGRAPH.JUSTIFY,
        }

        paragraph.alignment = alignment_map.get(
            align, WD_ALIGN_PARAGRAPH.LEFT
        )
        return paragraph

    def add_table(
        self, document, data, include_header=True,
        header_bold=True, col_widths=None
    ):
        if not data:
            logger.warning("No data provided for the table.")
            return None

        cols = max(len(row) for row in data)
        if cols == 0:
            return None

        table = document.add_table(rows=0, cols=cols)
        table.style = "Table Grid"

        for row_index, row_data in enumerate(data):
            cells = table.add_row().cells

            for col_index in range(cols):
                value = (
                    row_data[col_index]
                    if col_index < len(row_data) else ""
                )
                cells[col_index].text = str(value)

                if include_header and row_index == 0 and header_bold:
                    for run in cells[col_index].paragraphs[0].runs:
                        run.bold = True

        if col_widths and len(col_widths) == cols:
            for row in table.rows:
                for index, width in enumerate(col_widths):
                    row.cells[index].width = Inches(width)

        return table

    def add_picture(self, document, image_path, width_inches=None):
        if not os.path.exists(image_path):
            logger.error("Image file not found: %s", image_path)
            return False

        try:
            if width_inches:
                document.add_picture(
                    image_path, width=Inches(width_inches)
                )
            else:
                document.add_picture(image_path)

            return True

        except Exception as error:
            logger.exception("Error adding picture: %s", error)
            return False

    def save_document(self, document, output_path):
        try:
            ensure_directory_exists(
                os.path.dirname(output_path) or "."
            )
            document.save(output_path)
            logger.info("Word document saved: %s", output_path)
            return True

        except Exception as error:
            logger.exception("Error saving Word document: %s", error)
            return False

    def read_document(self, input_path):
        """อ่านย่อหน้าและตารางจากเอกสาร Word"""
        if not os.path.exists(input_path):
            logger.error("Word document not found: %s", input_path)
            return None

        document = docx.Document(input_path)

        paragraphs = [
            paragraph.text for paragraph in document.paragraphs
        ]

        tables = []
        for table in document.tables:
            table_data = []
            for row in table.rows:
                table_data.append([cell.text for cell in row.cells])
            tables.append(table_data)

        return {"paragraphs": paragraphs, "tables": tables}

Writing /content/document-automator/src/word_processor.py


In [6]:
%%writefile /content/document-automator/main.py
from pathlib import Path
from datetime import date

from PyPDF2 import PdfReader

from src.pdf_processor import PDFProcessor
from src.word_processor import WordProcessor
from src.utils import setup_logging, ensure_directory_exists

BASE_DIR = Path(__file__).resolve().parent
DOCS_DIR = BASE_DIR / "documents"

logger = setup_logging(__name__)


def main():
    ensure_directory_exists(str(DOCS_DIR))

    input_pdf = DOCS_DIR / "input_article.pdf"
    output_docx = DOCS_DIR / "output_report.docx"
    summary_docx = DOCS_DIR / "summary_report.docx"
    merged_pdf = DOCS_DIR / "merged_output.pdf"

    if not input_pdf.exists():
        logger.critical("Required input PDF not found: %s", input_pdf)
        return

    pdf_processor = PDFProcessor()
    word_processor = WordProcessor()

    logger.info("--- Starting Document Automation Workflow ---")

    # Task 1: Extract PDF text
    logger.info("Task 1: Extracting text from PDF.")
    extracted_text = pdf_processor.extract_text_from_pdf(str(input_pdf))

    if extracted_text is None or not extracted_text.strip():
        logger.error("No extractable text was found.")
        return

    reader = PdfReader(str(input_pdf))
    total_pages = len(reader.pages)

    # Task 2: Generate the main Word report
    logger.info("Task 2: Generating Word report.")
    document = word_processor.create_document()

    word_processor.add_heading(
        document, "Automated Document Report", level=1
    )

    word_processor.add_paragraph(
        document,
        "This report was automatically generated from PDF content.",
        bold=True,
        font_size=12
    )

    word_processor.add_paragraph(
        document,
        "--- Extracted Content from PDF ---",
        italic=True,
        font_size=10,
        align="center"
    )

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""

        word_processor.add_heading(
            document, f"PDF Page {page_number}", level=2
        )

        word_processor.add_paragraph(
            document,
            page_text if page_text.strip() else
            "[No selectable text found on this page.]"
        )

    word_processor.add_heading(document, "Summary Data", level=2)

    table_data = [
        ["Metric", "Value", "Notes"],
        ["Total Pages Processed", str(total_pages), "From input PDF"],
        ["Document Type", "PDF to Word", "Automated conversion"],
        ["Generated Date", date.today().isoformat(), "System date"],
    ]

    word_processor.add_table(
        document,
        table_data,
        include_header=True,
        header_bold=True,
        col_widths=[1.8, 1.5, 2.5]
    )

    if not word_processor.save_document(document, str(output_docx)):
        logger.error("Could not save main report.")

    # Task 3: Generate a separate summary Word document
    logger.info("Task 3: Generating summary Word document.")
    summary = word_processor.create_document()

    word_processor.add_heading(
        summary, "PDF Content Summary", level=1
    )

    word_processor.add_paragraph(
        summary,
        f"Source file: {input_pdf.name}"
    )

    word_processor.add_paragraph(
        summary,
        f"Total pages processed: {total_pages}"
    )

    word_processor.add_heading(
        summary, "Extracted Text Overview", level=2
    )

    # Keep a readable excerpt for the summary
    excerpt = extracted_text.replace("--- Page End:", "")
    excerpt = excerpt[:2000]

    word_processor.add_paragraph(summary, excerpt)

    word_processor.add_heading(
        summary, "Processing Information", level=2
    )

    word_processor.add_table(
        summary,
        [
            ["Item", "Result"],
            ["PDF Extraction", "Completed"],
            ["Word Report", output_docx.name],
            ["Summary Report", summary_docx.name],
            ["Generated Date", date.today().isoformat()],
        ]
    )

    word_processor.save_document(summary, str(summary_docx))

    # Task 4: Merge the two dummy PDFs
    logger.info("Task 4: Merging dummy PDFs.")

    dummy_pdfs = [
        DOCS_DIR / "dummy_doc1.pdf",
        DOCS_DIR / "dummy_doc2.pdf",
    ]

    if all(path.exists() for path in dummy_pdfs):
        merged = pdf_processor.merge_pdfs(
            [str(path) for path in dummy_pdfs],
            str(merged_pdf)
        )

        if merged:
            logger.info("PDF merge completed: %s", merged_pdf)
        else:
            logger.warning("PDF merge did not complete.")
    else:
        logger.warning("Dummy PDFs not found; skipping merge.")

    # Task 5: Split the source PDF into one-page files
    logger.info("Task 5: Splitting the source PDF.")
    split_dir = DOCS_DIR / "split_pages"

    pdf_processor.split_pdf(
        str(input_pdf),
        output_dir=str(split_dir),
        pages_per_split=1
    )

    logger.info("--- Document Automation Workflow Completed ---")

    print("\nAll tasks finished.")
    print("Main report:", output_docx)
    print("Summary report:", summary_docx)
    print("Merged PDF:", merged_pdf)
    print("Split PDF folder:", split_dir)


if __name__ == "__main__":
    main()

Writing /content/document-automator/main.py


In [7]:
%cd /content/document-automator
!python main.py

/content/document-automator
2026-10-11 04:48:45,433 - src.pdf_processor - INFO - PDFProcessor initialized.
2026-10-11 04:48:45,433 - src.word_processor - INFO - WordProcessor initialized.
2026-10-11 04:48:45,433 - __main__ - INFO - --- Starting Document Automation Workflow ---
2026-10-11 04:48:45,433 - __main__ - INFO - Task 1: Extracting text from PDF.
2026-10-11 04:48:45,438 - src.pdf_processor - INFO - Extracted text from /content/document-automator/documents/input_article.pdf (2 pages).
2026-10-11 04:48:45,439 - __main__ - INFO - Task 2: Generating Word report.
2026-10-11 04:48:45,520 - src.word_processor - INFO - Word document saved: /content/document-automator/documents/output_report.docx
2026-10-11 04:48:45,521 - __main__ - INFO - Task 3: Generating summary Word document.
2026-10-11 04:48:45,586 - src.word_processor - INFO - Word document saved: /content/document-automator/documents/summary_report.docx
2026-10-11 04:48:45,586 - __main__ - INFO - Task 4: Merging dummy PDFs.
2026-

In [8]:
from pathlib import Path

docs_dir = Path("/content/document-automator/documents")

print("Files generated:\n")

for path in sorted(docs_dir.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(docs_dir)} ({path.stat().st_size:,} bytes)")

Files generated:

dummy_doc1.pdf (1,407 bytes)
dummy_doc2.pdf (1,408 bytes)
input_article.pdf (2,331 bytes)
merged_output.pdf (1,660 bytes)
output_report.docx (37,371 bytes)
split_pages/input_article_part_1.pdf (1,141 bytes)
split_pages/input_article_part_2.pdf (1,251 bytes)
summary_report.docx (37,327 bytes)


In [9]:
from src.word_processor import WordProcessor

processor = WordProcessor()

report_path = docs_dir / "output_report.docx"
result = processor.read_document(str(report_path))

if result:
    print("=== Paragraphs ===")
    for paragraph in result["paragraphs"]:
        if paragraph.strip():
            print(paragraph)

    print("\n=== Tables ===")
    for table_index, table in enumerate(result["tables"], start=1):
        print(f"\nTable {table_index}")
        for row in table:
            print(row)

2026-10-11 04:48:45,860 - src.word_processor - INFO - WordProcessor initialized.


=== Paragraphs ===
Automated Document Report
This report was automatically generated from PDF content.
--- Extracted Content from PDF ---
PDF Page 1
Introduction to Automation
Automation uses technology to perform tasks
with minimal human intervention.
It improves efficiency, reliability, and speed.
Automation can increase productivity, reduce costs,
and improve the quality of work.

PDF Page 2
Benefits of Automation
1. Increased Efficiency: Tasks are completed faster.
2. Cost Reduction: Lower labor costs and less waste.
3. Improved Accuracy: Minimizes human error.
4. Enhanced Safety: Automates dangerous tasks.
5. Greater Consistency: Provides uniform quality.
Conclusion
Automation is transforming industries globally.

Summary Data

=== Tables ===

Table 1
['Metric', 'Value', 'Notes']
['Total Pages Processed', '2', 'From input PDF']
['Document Type', 'PDF to Word', 'Automated conversion']
['Generated Date', '2026-10-11', 'System date']


In [10]:
from google.colab import files

files_to_download = {
    "Word Report": docs_dir / "output_report.docx",
    "Summary Report": docs_dir / "summary_report.docx",
    "Merged PDF": docs_dir / "merged_output.pdf",
}

print("เลือกไฟล์ที่ต้องการดาวน์โหลด:")

เลือกไฟล์ที่ต้องการดาวน์โหลด:


In [11]:
from google.colab import files

files.download(
    "/content/document-automator/documents/merged_output.pdf"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from google.colab import files

files.download(
    "/content/document-automator/documents/output_report.docx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>